# Serve Qwen-Coder on Kaggle (issue #2)

Free GPU option when ASU Sol is busy. **Settings → Accelerator → GPU T4 x2**, and **Internet → On**.
Kaggle gives 30 GPU hours per week; a session lasts up to 12 hours.

| Model | Fits on 2× T4 (2 × 15 GB)? |
|---|---|
| `qwen2.5-coder:7b` (Ollama, 4-bit, ~5 GB) | yes |
| `qwen2.5-coder:32b` (Ollama, 4-bit, ~20 GB) | yes, Ollama splits it across both GPUs |

Run the cells top to bottom. The last cell measures throughput (PRs/hour) for issue #2.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the server in the background
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_KEEP_ALIVE="24h", OLLAMA_HOST="0.0.0.0:11434")
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("ollama is up"); break
    except Exception:
        time.sleep(1)

In [ ]:
# 3. Download the model (7B ~5 GB, 32B ~20 GB)
MODEL = "qwen2.5-coder:7b"      # or "qwen2.5-coder:32b"
!ollama pull {MODEL}

In [ ]:
# 4. Get the team code (for the client + throughput script)
!git clone -q https://github.com/pranjalpadakannaya/SignalPR.git /kaggle/working/SignalPR
%cd /kaggle/working/SignalPR
!pip -q install pyyaml

In [ ]:
# 5. Smoke test through the shared client
import sys; sys.path.insert(0, "/kaggle/working/SignalPR")
from agent.llm_client import LLMClient, LLMConfig
llm = LLMClient(LLMConfig.from_preset("qwen-7b" if "7b" in MODEL else "qwen-32b", backend="ollama",
                                      cache_dir="/kaggle/working/llm_cache"))
print(llm.health())
print(llm.complete("Reply with one word: ready?").text)

In [ ]:
# 6. Throughput for issue #2 (writes results/llm/throughput.csv; download it from the Output tab)
!python scripts/llm_throughput.py --backend ollama --model {MODEL} --n 24 --concurrency 4 --where kaggle-2xT4

### Optional: call this server from your laptop
Expose it with a free Cloudflare quick tunnel, then set `SIGNALPR_LLM_BASE_URL=https://<random>.trycloudflare.com/v1`.
```
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:11434 > tunnel.log 2>&1 &
!sleep 8 && grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1
```
The URL changes every session and anyone with it can use your model, so stop the notebook when you are done.